## Concept 8 — LangGraph Checkpointing & Persistent State

### What is it?
The modern production approach. LangGraph treats memory as **first-class agent state** — the entire
conversation is automatically saved and restored via a Checkpointer. No manual `save_context()` calls.

### Two Layers
```
SHORT-TERM (per session):
  thread_id  → Checkpointer auto-saves all messages
  Same thread_id = same conversation, always

LONG-TERM (cross-session):
  Long-term Store → agent explicitly saves/retrieves user facts across threads
```

### Why It Beats Manual Memory
No glue code, works in multi-agent flows, swap MemorySaver -> PostgresSaver for production.

In [ ]:
print('hi')

In [ ]:
# Install langgraph if not present (not in requirement.txt by default)
# !pip install langgraph

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

from MemoryUtils import get_llm, TEST_TURNS, TEST_QUERIES, run_recall_test
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore

llm = get_llm()
print('LLM and LangGraph ready')

### Step 1 — Build a Graph with Short-Term Memory (Checkpointer)
`MemorySaver` auto-saves all messages per `thread_id`. No `save_context()` needed.

In [ ]:
checkpointer = MemorySaver()

def assistant_node(state):
    """Simple assistant node — just calls the LLM with accumulated message history."""
    response = llm.invoke(state['messages'])
    return {'messages': [response]}

builder = StateGraph(MessagesState)
builder.add_node('assistant', assistant_node)
builder.set_entry_point('assistant')
builder.set_finish_point('assistant')

# Compile with checkpointer — this is all that is needed for auto-memory
short_term_app = builder.compile(checkpointer=checkpointer)

print('Graph compiled with MemorySaver checkpointer.')
print('thread_id controls which conversation is loaded on each invocation.')

### Step 2 — Feed Test Turns Using Thread ID
All messages are auto-persisted. Same `thread_id` = same conversation continued.

In [ ]:
THREAD_ID = 'mohan_session_1'
config = {'configurable': {'thread_id': THREAD_ID}}

system_msg = SystemMessage(content='You are a helpful assistant. Use the conversation history to answer accurately.')

# Feed all 5 test turns into the graph
for human, ai in TEST_TURNS:
    # Invoke with human message — graph accumulates all messages automatically
    short_term_app.invoke(
        {'messages': [HumanMessage(content=human)]},
        config
    )

# Check how many messages are stored
state = short_term_app.get_state(config)
messages = state.values.get('messages', [])
print(f'Messages stored in checkpointer for thread "{THREAD_ID}": {len(messages)}')
print('All messages are auto-persisted — no save_context() call needed.')

### Step 3 — Demonstrate Short-Term Recall
Query the graph with the same thread_id — it automatically loads and uses the full history.

In [ ]:
print('Recall test for thread:', THREAD_ID)
print('=' * 55)

for query in TEST_QUERIES[:3]:
    result = short_term_app.invoke(
        {'messages': [HumanMessage(content=query)]},
        config
    )
    answer = result['messages'][-1].content
    print(f'Q: {query}')
    print(f'A: {answer}')
    print('-' * 40)

### Step 4 — Different Thread = Fresh Conversation
Same graph, different `thread_id` — completely isolated, no memory of the previous session.

In [ ]:
new_config = {'configurable': {'thread_id': 'brand_new_session'}}

result = short_term_app.invoke(
    {'messages': [HumanMessage(content='What is my name?')]},
    new_config
)
answer = result['messages'][-1].content

print('Different thread_id: "brand_new_session"')
print('Q: What is my name?')
print(f'A: {answer}')
print()
print('Expected: LLM says it does not know — no history in this thread.')

### Step 5 — Long-Term Memory with InMemoryStore
The Store persists user-level facts ACROSS threads (sessions). Agent explicitly saves and retrieves.

In [ ]:
long_term_store = InMemoryStore()
lt_checkpointer = MemorySaver()

def assistant_with_longterm(state, config, *, store):
    """Assistant that reads from and writes to the long-term store."""
    user_id = config.get('configurable', {}).get('user_id', 'default_user')

    # Load long-term memories for this user
    memories = store.search(('users', user_id))
    memory_lines = [m.value.get('fact', '') for m in memories]
    memory_text  = '\n'.join(memory_lines) if memory_lines else 'No prior knowledge about this user.'

    system = SystemMessage(content=(
        f'You are a helpful assistant.\n'
        f'Long-term user knowledge:\n{memory_text}'
    ))

    response = llm.invoke([system] + state['messages'])

    # Save important facts to long-term store (in real use, decide dynamically)
    last_human = state['messages'][-1].content if state['messages'] else ''
    if any(kw in last_human.lower() for kw in ['my name', 'i am', 'i use', 'i work', 'i build']):
        fact_key = f'fact_{len(memories)}'
        store.put(('users', user_id), fact_key, {'fact': last_human})

    return {'messages': [response]}

lt_builder = StateGraph(MessagesState)
lt_builder.add_node('assistant', assistant_with_longterm)
lt_builder.set_entry_point('assistant')
lt_builder.set_finish_point('assistant')

lt_app = lt_builder.compile(checkpointer=lt_checkpointer, store=long_term_store)
print('Long-term memory graph compiled.')

### Step 6 — Session 1: Learn user facts (Thread A)
Feed turns in Session 1. Facts get saved to long-term store.

In [ ]:
session1_config = {'configurable': {'thread_id': 'session_1', 'user_id': 'mohan'}}

for human, _ in TEST_TURNS[:3]:
    lt_app.invoke({'messages': [HumanMessage(content=human)]}, session1_config)

saved_facts = long_term_store.search(('users', 'mohan'))
print(f'Long-term store now has {len(saved_facts)} fact(s) for user "mohan":')
for f in saved_facts:
    print(f'  - {f.value["fact"]}')

### Step 7 — Session 2: Recall from long-term store (Thread B)
Brand new thread_id (fresh conversation) but same user_id — long-term facts are loaded automatically.

In [ ]:
session2_config = {'configurable': {'thread_id': 'session_2', 'user_id': 'mohan'}}

result = lt_app.invoke(
    {'messages': [HumanMessage(content='What is my name and what am I building?')]},
    session2_config
)
answer = result['messages'][-1].content

print('Session 2 (brand new thread, same user_id):')
print('Q: What is my name and what am I building?')
print(f'A: {answer}')
print()
print('The assistant knows, because long-term store carries facts across sessions.')

### Full Function — chat_with_langgraph

In [ ]:
# Re-use short_term_app with THREAD_ID which already has all 5 test turns loaded

def chat_with_langgraph(query):
    result = short_term_app.invoke(
        {'messages': [HumanMessage(content=query)]},
        config   # THREAD_ID config from Step 2
    )
    return result['messages'][-1].content

### Test — Memory Recall via LangGraph Checkpointing

In [ ]:
run_recall_test(chat_with_langgraph, label='Concept 8 -- LangGraph Checkpointing')

### Production Backends
Replace `MemorySaver` with a persistent backend:

```python
# SQLite (persistent across restarts, zero infra)
from langgraph.checkpoint.sqlite import SqliteSaver
checkpointer = SqliteSaver.from_conn_string('memory.db')

# PostgreSQL (production, multi-server)
from langgraph.checkpoint.postgres import PostgresSaver
checkpointer = PostgresSaver.from_conn_string('postgresql://...')
```

Everything else stays exactly the same — only the checkpointer line changes.